# HAR — Faithful re-implementation of Xia et al. (2020) LSTM-CNN
Paper: *LSTM-CNN Architecture for Human Activity Recognition*, IEEE Access 8, 2020 (reported **95.78%** on UCI HAR).

**Paper recipe (what this notebook follows):** raw 9-channel inertial windows (128 x 9) -> 2 stacked LSTM layers (32 units each) -> Conv1D(64, k=3) -> MaxPool -> Conv1D(128, k=3) -> MaxPool -> Global Average Pooling -> Batch Norm -> Softmax. No hand-made features, no attention. Adam, official UCI train/test split.

> Caveat: I could not open the paper's full text from here, so layer sizes/hyperparameters are from my recollection of it and its public re-implementations — check them against the PDF (Table of architecture) and edit the `CFG` dict below.

**Honest evaluation:** the paper-style protocol (pick the best epoch by looking at the test set) is optimistic. This notebook reports BOTH:
1. **Strict** — trained for fixed epochs on the full training set, test touched once at the end (the number to put in your report).
2. **Paper-style / optimistic** — best test accuracy over epochs (peeks at test; shown only to explain the gap to 95.78%).

In [ ]:
!pip -q install tensorflow scikit-learn matplotlib seaborn pandas
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, seaborn as sns, os, zipfile, time
from tensorflow.keras import layers, models, Input, regularizers
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

CFG = dict(lstm_units=32, filters=(64,128), kernel=3, l2=1e-4, lr=1e-3,
           batch=64, epochs=100, seeds=[1,2,3,4,5], noise_std=0.0)
np.random.seed(0); tf.random.set_seed(0)

In [ ]:
from google.colab import files
up = files.upload(); ZIP = list(up.keys())[0]
zipfile.ZipFile(ZIP).extractall('har_data')
inner='har_data/UCI HAR Dataset.zip'
if os.path.exists(inner): zipfile.ZipFile(inner).extractall('har_data')
BASE='har_data/UCI HAR Dataset'

SIGNALS=["body_acc_x","body_acc_y","body_acc_z","body_gyro_x","body_gyro_y","body_gyro_z","total_acc_x","total_acc_y","total_acc_z"]
NAMES=["WALKING","WALKING_UPSTAIRS","WALKING_DOWNSTAIRS","SITTING","STANDING","LAYING"]
def load(s):
    X=np.stack([np.loadtxt(f"{BASE}/{s}/Inertial Signals/{g}_{s}.txt") for g in SIGNALS],-1)
    y=np.loadtxt(f"{BASE}/{s}/y_{s}.txt").astype(int)-1
    return X.astype('float32'),y
X_tr,y_tr=load('train'); X_te,y_te=load('test')
mu=X_tr.mean((0,1),keepdims=True); sd=X_tr.std((0,1),keepdims=True)+1e-8   # train-only stats
X_tr=(X_tr-mu)/sd; X_te=(X_te-mu)/sd
print(X_tr.shape,X_te.shape)

In [ ]:
def build_xia(cfg=CFG):
    reg=regularizers.l2(cfg['l2'])
    i=Input((128,9))
    x=layers.LSTM(cfg['lstm_units'],return_sequences=True,kernel_regularizer=reg)(i)
    x=layers.LSTM(cfg['lstm_units'],return_sequences=True,kernel_regularizer=reg)(x)
    x=layers.Conv1D(cfg['filters'][0],cfg['kernel'],padding='same',activation='relu',kernel_regularizer=reg)(x)
    x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(cfg['filters'][1],cfg['kernel'],padding='same',activation='relu',kernel_regularizer=reg)(x)
    x=layers.MaxPooling1D(2)(x)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.BatchNormalization()(x)
    o=layers.Dense(6,activation='softmax')(x)
    m=models.Model(i,o,name='Xia_LSTM_CNN')
    m.compile(tf.keras.optimizers.Adam(cfg['lr']),'sparse_categorical_crossentropy',['accuracy'])
    return m
build_xia().summary()

## Train — several seeds, fixed epochs (strict) + per-epoch test curve (paper-style)

In [ ]:
class TestTracker(tf.keras.callbacks.Callback):
    # records test acc each epoch ONLY for the optimistic 'paper-style' number; never used for model selection
    def on_epoch_end(self,e,logs=None):
        self.accs=getattr(self,'accs',[]); self.accs.append(self.model.evaluate(X_te,y_te,verbose=0)[1])

runs=[]; probs=[]
for s in CFG['seeds']:
    tf.keras.utils.set_random_seed(s)
    m=build_xia(); tr=TestTracker()
    m.fit(X_tr,y_tr,epochs=CFG['epochs'],batch_size=CFG['batch'],verbose=0,callbacks=[tr])
    p=m.predict(X_te,verbose=0); probs.append(p)
    final=accuracy_score(y_te,p.argmax(1))
    runs.append(dict(seed=s,strict_final_epoch=final,optimistic_best_epoch=max(tr.accs),best_epoch=int(np.argmax(tr.accs))+1))
    print(runs[-1])
df=pd.DataFrame(runs); display(df)
print(f"Strict (final epoch), mean±std over seeds: {df.strict_final_epoch.mean():.4f} ± {df.strict_final_epoch.std():.4f}")
print(f"Optimistic (best-on-test epoch), mean:     {df.optimistic_best_epoch.mean():.4f}   <- paper-style, leaky")

## Strict evaluation of the seed ensemble (average softmax, final epoch, test used once)

In [ ]:
P=np.mean(probs,0); yp=P.argmax(1)
acc=accuracy_score(y_te,yp); pr,rc,f1,_=precision_recall_fscore_support(y_te,yp,average='macro')
print(f"ENSEMBLE x{len(probs)}  accuracy={acc:.4f}  precision={pr:.4f}  recall={rc:.4f}  macroF1={f1:.4f}")
print(classification_report(y_te,yp,target_names=NAMES,digits=3))
plt.figure(figsize=(7,6)); sns.heatmap(confusion_matrix(y_te,yp),annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES)
plt.title('Xia-style LSTM-CNN ensemble — confusion matrix'); plt.tight_layout(); plt.savefig('confusion_xia.png',dpi=150); plt.show()

## Reading the result
* The **strict** ensemble number is the defensible one. Expect roughly 92–95%.
* If the *optimistic* number approaches ~95–96%, that explains the paper's 95.78%: it is a best-epoch-on-test figure.
* SITTING vs STANDING is the main error source on this dataset (near-identical accelerometer signals); every raw-signal model struggles there.